# Week 5 EDA: Feature Design for Recipe Recommendation

This notebook performs exploratory data analysis (EDA) to guide feature representation and dimensionality reduction for the Food.com dataset. The goal is to justify Week 5 feature choices for numeric, text/list, categorical, and temporal data, while keeping recipe-level content features separate from review-level interaction features.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast
import re

DATA_DIR = Path("../data/processed")
ARTIFACT_DIR = Path("../artifacts/week5/eda_summaries")
FIGURE_DIR = Path("../reports/figures")

for p in [ARTIFACT_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def save_df(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

def summarize_missingness(df):
    summary = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(t) for t in df.dtypes],
        "non_null_count": df.notna().sum().values,
        "null_count": df.isna().sum().values,
        "pct_missing": (df.isna().mean() * 100).values,
        "unique_count": df.nunique(dropna=True).values,
    })
    return summary.sort_values("pct_missing", ascending=False)

def find_column(df, candidates):
    col_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in col_map:
            return col_map[cand.lower()]
    return None

## Load processed datasets
We load the processed recipes (catalog) and reviews (interaction) tables and review their shapes and columns.

In [2]:
recipes_path = DATA_DIR / "recipes_processed.csv"
reviews_path = DATA_DIR / "reviews_processed.csv"

recipes = pd.read_csv(recipes_path)
reviews = pd.read_csv(reviews_path)

print("Recipes shape:", recipes.shape)
print("Reviews shape:", reviews.shape)
print("Recipes columns:")
print(recipes.columns.tolist())
print("Reviews columns:")
print(reviews.columns.tolist())

print("Recipes head:")
display(recipes.head())
print("Reviews head:")
display(reviews.head())

Recipes shape: (522517, 33)
Reviews shape: (1401982, 8)
Recipes columns:
['RecipeId', 'Name', 'AuthorId', 'AuthorName', 'CookTime', 'PrepTime', 'TotalTime', 'DatePublished', 'Description', 'Images', 'RecipeCategory', 'Keywords', 'RecipeIngredientQuantities', 'RecipeIngredientParts', 'AggregatedRating', 'ReviewCount', 'Calories', 'FatContent', 'SaturatedFatContent', 'CholesterolContent', 'SodiumContent', 'CarbohydrateContent', 'FiberContent', 'SugarContent', 'ProteinContent', 'RecipeServings', 'RecipeYield', 'RecipeInstructions', 'CookTime_Minutes', 'PrepTime_Minutes', 'TotalTime_Minutes', 'NumIngredients', 'NumQuantities']
Reviews columns:
['ReviewId', 'RecipeId', 'AuthorId', 'AuthorName', 'Rating', 'Review', 'DateSubmitted', 'DateModified']
Recipes head:


,RecipeId,Name,AuthorId,AuthorName,CookTime,PrepTime,TotalTime,DatePublished,Description,Images,...,SugarContent,ProteinContent,RecipeServings,RecipeYield,RecipeInstructions,CookTime_Minutes,PrepTime_Minutes,TotalTime_Minutes,NumIngredients,NumQuantities
0,38,Low-Fat Berry Blue Frozen Dessert,1533,Dancer,PT24H,PT45M,PT24H45M,1999-08-09T21:46:00Z,Make and share this Low-Fat Berry Blue Frozen ...,['https://img.sndimg.com/food/image/upload/w_5...,...,30.2,3.2,4.0,NaN,"['Toss 2 cups berries with sugar.', 'Let stand...",1440.0,45.0,1485.0,4,4
1,39,Biryani,1567,elly9812,PT25M,PT4H,PT4H25M,1999-08-29T13:12:00Z,Make and share this Biryani recipe from Food.com.,['https://img.sndimg.com/food/image/upload/w_5...,...,20.4,63.4,6.0,NaN,['Soak saffron in warm milk for 5 minutes and ...,25.0,240.0,265.0,25,26
2,40,Best Lemonade,1566,Stephen Little,PT5M,PT30M,PT35M,1999-09-05T19:52:00Z,This is from one of my first Good House Keepi...,['https://img.sndimg.com/food/image/upload/w_5...,...,77.2,0.3,4.0,NaN,"['Into a 1 quart Jar with tight fitting lid, p...",5.0,30.0,35.0,5,6
3,41,Carina's Tofu-Vegetable Kebabs,1586,Cyclopz,PT20M,PT24H,PT24H20M,1999-09-03T14:54:00Z,This dish is best prepared a day in advance to...,['https://img.sndimg.com/food/image/upload/w_5...,...,32.1,29.3,2.0,4 kebabs,"['Drain the tofu, carefully squeezing out exce...",20.0,1440.0,1460.0,14,15
4,42,Cabbage Soup,1538,Duckie067,PT30M,PT20M,PT50M,1999-09-19T06:19:00Z,Make and share this Cabbage Soup recipe from F...,['https://img.sndimg.com/food/image/upload/w_5...,...,17.7,4.3,4.0,NaN,['Mix everything together and bring to a boil....,30.0,20.0,50.0,5,5


Reviews head:


,ReviewId,RecipeId,AuthorId,AuthorName,Rating,Review,DateSubmitted,DateModified
0,2,992,2008,gayg msft,5,better than any you can get at a restaurant!,2000-01-25T21:44:00Z,2000-01-25T21:44:00Z
1,7,4384,1634,Bill Hilbrich,4,"I cut back on the mayo, and made up the differ...",2001-10-17T16:49:59Z,2001-10-17T16:49:59Z
2,9,4523,2046,Gay Gilmore ckpt,2,i think i did something wrong because i could ...,2000-02-25T09:00:00Z,2000-02-25T09:00:00Z
3,13,7435,1773,Malarkey Test,5,easily the best i have ever had. juicy flavor...,2000-03-13T21:15:00Z,2000-03-13T21:15:00Z
4,14,44,2085,Tony Small,5,An excellent dish.,2000-03-28T12:51:00Z,2000-03-28T12:51:00Z


## Data quality and granularity audit
We confirm the granularity and key identifiers in recipes and reviews, and check for orphans and duplicates.

In [3]:
recipe_id_col = find_column(recipes, ["RecipeId", "RecipeID", "recipe_id"])
review_id_col = find_column(reviews, ["ReviewId", "ReviewID", "review_id"])
author_id_col = find_column(reviews, ["AuthorId", "AuthorID", "author_id", "UserId", "UserID"])
review_recipe_id_col = find_column(reviews, ["RecipeId", "RecipeID", "recipe_id"])

recipes_rows, recipes_cols = recipes.shape
reviews_rows, reviews_cols = reviews.shape

unique_recipe_ids = recipes[recipe_id_col].nunique() if recipe_id_col else np.nan
duplicate_recipe_ids = recipes_rows - unique_recipe_ids if recipe_id_col else np.nan

unique_review_ids = reviews[review_id_col].nunique() if review_id_col else np.nan
duplicate_review_ids = reviews_rows - unique_review_ids if review_id_col else np.nan

unique_reviewers = reviews[author_id_col].nunique() if author_id_col else np.nan
unique_reviewed_recipes = reviews[review_recipe_id_col].nunique() if review_recipe_id_col else np.nan

orphans = np.nan
if recipe_id_col and review_recipe_id_col:
    recipe_ids = set(recipes[recipe_id_col].dropna().astype(str))
    review_recipe_ids = set(reviews[review_recipe_id_col].dropna().astype(str))
    orphans = len(review_recipe_ids - recipe_ids)

recipes_mem_mb = recipes.memory_usage(deep=True).sum() / (1024 ** 2)
reviews_mem_mb = reviews.memory_usage(deep=True).sum() / (1024 ** 2)

quality_summary = pd.DataFrame([
    {"metric": "recipes_rows", "value": recipes_rows},
    {"metric": "recipes_cols", "value": recipes_cols},
    {"metric": "reviews_rows", "value": reviews_rows},
    {"metric": "reviews_cols", "value": reviews_cols},
    {"metric": "unique_recipe_ids", "value": unique_recipe_ids},
    {"metric": "duplicate_recipe_ids", "value": duplicate_recipe_ids},
    {"metric": "unique_review_ids", "value": unique_review_ids},
    {"metric": "duplicate_review_ids", "value": duplicate_review_ids},
    {"metric": "unique_reviewers", "value": unique_reviewers},
    {"metric": "unique_reviewed_recipes", "value": unique_reviewed_recipes},
    {"metric": "orphan_review_recipe_ids", "value": orphans},
    {"metric": "recipes_mem_mb", "value": round(recipes_mem_mb, 2)},
    {"metric": "reviews_mem_mb", "value": round(reviews_mem_mb, 2)}
])

save_df(quality_summary, ARTIFACT_DIR / "data_quality_summary.csv")
display(quality_summary)

,metric,value
0,recipes_rows,522517.00
1,recipes_cols,33.00
2,reviews_rows,1401982.00
3,reviews_cols,8.00
4,unique_recipe_ids,522517.00
5,duplicate_recipe_ids,0.00
6,unique_review_ids,1401982.00
7,duplicate_review_ids,0.00
8,unique_reviewers,271907.00
9,unique_reviewed_recipes,271678.00


**Interpretation**
- If recipe IDs are unique and duplicates are near zero, recipe-level matrices can be built safely without duplicating rows.
- Reviews should remain separate because they are interaction records and will expand recipe rows if joined.
- Orphan review counts indicate whether the review table has references not present in the catalog.

## Missingness audit
We quantify missingness and uniqueness for recipes and reviews separately.

In [4]:
recipes_missingness = summarize_missingness(recipes)
reviews_missingness = summarize_missingness(reviews)

save_df(recipes_missingness, ARTIFACT_DIR / "recipes_missingness.csv")
save_df(reviews_missingness, ARTIFACT_DIR / "reviews_missingness.csv")

print("Top missing recipe columns:")
display(recipes_missingness.head(10))
print("Top missing review columns:")
display(reviews_missingness.head(10))

Top missing recipe columns:


,column,dtype,non_null_count,null_count,pct_missing,unique_count
26,RecipeYield,str,174446,348071,66.614292,34043
14,AggregatedRating,float64,269294,253223,48.462155,9
15,ReviewCount,float64,275028,247489,47.364775,420
25,RecipeServings,float64,339606,182911,35.005751,171
4,CookTime,str,439972,82545,15.797572,490
28,CookTime_Minutes,float64,439972,82545,15.797572,490
10,RecipeCategory,str,521766,751,0.143727,311
8,Description,str,522512,5,0.000957,492838
29,PrepTime_Minutes,float64,522515,2,0.000383,316
30,TotalTime_Minutes,float64,522516,1,0.000191,1239


Top missing review columns:


,column,dtype,non_null_count,null_count,pct_missing,unique_count
5,Review,str,1401768,214,0.015264,1392745
0,ReviewId,int64,1401982,0,0.000000,1401982
1,RecipeId,int64,1401982,0,0.000000,271678
2,AuthorId,int64,1401982,0,0.000000,271907
3,AuthorName,str,1401982,0,0.000000,241365
4,Rating,int64,1401982,0,0.000000,6
6,DateSubmitted,str,1401982,0,0.000000,1384268
7,DateModified,str,1401982,0,0.000000,1384268


**Interpretation**
- Columns with high missingness may require imputation, exclusion, or separate handling.
- Unique counts highlight high-cardinality fields that should not be one-hot encoded directly.

## Recipe Servings - Recipe Yields/ Preptime minutes, Cooktime minutes and Total minutes checkpoint 

In [6]:
# ============================================================
# 1. RecipeServings vs RecipeYield presence analysis
# ============================================================

servings_col = "RecipeServings"
yield_col = "RecipeYield"

# Basic validation
required_cols = [servings_col, yield_col]
missing_cols = [col for col in required_cols if col not in recipes.columns]

if missing_cols:
    raise ValueError(f"Missing columns in recipes dataframe: {missing_cols}")

total_recipes = len(recipes)

# Treat empty strings in RecipeYield as missing too
servings_present = recipes[servings_col].notna()

yield_clean = recipes[yield_col].astype("string").str.strip()
yield_present = recipes[yield_col].notna() & yield_clean.ne("")

servings_only = servings_present & ~yield_present
yield_only = ~servings_present & yield_present
both_present = servings_present & yield_present
both_missing = ~servings_present & ~yield_present

servings_yield_summary = pd.DataFrame(
    {
        "count": [
            servings_only.sum(),
            yield_only.sum(),
            both_present.sum(),
            both_missing.sum(),
            total_recipes,
        ],
        "percent_total_dataset": [
            servings_only.mean() * 100,
            yield_only.mean() * 100,
            both_present.mean() * 100,
            both_missing.mean() * 100,
            100.0,
        ],
        "interpretation": [
            "RecipeServings exists, RecipeYield missing",
            "RecipeYield exists, RecipeServings missing",
            "Both RecipeServings and RecipeYield exist",
            "Both RecipeServings and RecipeYield missing",
            "Total number of recipes",
        ],
    },
    index=[
        "servings_only",
        "yield_only",
        "both_present",
        "both_missing",
        "total_recipes",
    ],
)

servings_yield_summary["count"] = servings_yield_summary["count"].astype(int)
servings_yield_summary["percent_total_dataset"] = servings_yield_summary[
    "percent_total_dataset"
].round(4)

display(servings_yield_summary)

# Optional: save output
# servings_yield_summary.to_csv("artifacts/week5/eda_summaries/servings_yield_presence_summary.csv")


# ============================================================
# 2. Time fields presence and consistency analysis
# ============================================================

prep_col = "PrepTime_Minutes"
cook_col = "CookTime_Minutes"
total_col = "TotalTime_Minutes"

required_time_cols = [prep_col, cook_col, total_col]
missing_time_cols = [col for col in required_time_cols if col not in recipes.columns]

if missing_time_cols:
    raise ValueError(f"Missing time columns in recipes dataframe: {missing_time_cols}")

# Coerce to numeric just in case
time_df = recipes[[prep_col, cook_col, total_col]].copy()

for col in required_time_cols:
    time_df[col] = pd.to_numeric(time_df[col], errors="coerce")

prep_present = time_df[prep_col].notna()
cook_present = time_df[cook_col].notna()
total_present = time_df[total_col].notna()

# Presence patterns
all_three_present = prep_present & cook_present & total_present
prep_cook_present_total_missing = prep_present & cook_present & ~total_present
prep_total_present_cook_missing = prep_present & ~cook_present & total_present
cook_total_present_prep_missing = ~prep_present & cook_present & total_present
only_prep_present = prep_present & ~cook_present & ~total_present
only_cook_present = ~prep_present & cook_present & ~total_present
only_total_present = ~prep_present & ~cook_present & total_present
all_three_missing = ~prep_present & ~cook_present & ~total_present

time_presence_summary = pd.DataFrame(
    {
        "count": [
            all_three_present.sum(),
            prep_cook_present_total_missing.sum(),
            prep_total_present_cook_missing.sum(),
            cook_total_present_prep_missing.sum(),
            only_prep_present.sum(),
            only_cook_present.sum(),
            only_total_present.sum(),
            all_three_missing.sum(),
            total_recipes,
        ],
        "percent_total_dataset": [
            all_three_present.mean() * 100,
            prep_cook_present_total_missing.mean() * 100,
            prep_total_present_cook_missing.mean() * 100,
            cook_total_present_prep_missing.mean() * 100,
            only_prep_present.mean() * 100,
            only_cook_present.mean() * 100,
            only_total_present.mean() * 100,
            all_three_missing.mean() * 100,
            100.0,
        ],
        "interpretation": [
            "Prep, cook, and total time are all present",
            "Prep and cook exist, but total time is missing",
            "Prep and total exist, but cook time is missing",
            "Cook and total exist, but prep time is missing",
            "Only prep time exists",
            "Only cook time exists",
            "Only total time exists",
            "All three time fields are missing",
            "Total number of recipes",
        ],
    },
    index=[
        "all_three_present",
        "prep_cook_present_total_missing",
        "prep_total_present_cook_missing",
        "cook_total_present_prep_missing",
        "only_prep_present",
        "only_cook_present",
        "only_total_present",
        "all_three_missing",
        "total_recipes",
    ],
)

time_presence_summary["count"] = time_presence_summary["count"].astype(int)
time_presence_summary["percent_total_dataset"] = time_presence_summary[
    "percent_total_dataset"
].round(4)

display(time_presence_summary)

# Optional: save output
# time_presence_summary.to_csv("artifacts/week5/eda_summaries/time_presence_summary.csv")


# ============================================================
# 3. Time arithmetic consistency analysis
#    Checks whether TotalTime = PrepTime + CookTime
# ============================================================

# Use small tolerance because times are floats
tolerance = 1e-6

total_equals_prep_plus_cook = (
    all_three_present
    & np.isclose(
        time_df[total_col],
        time_df[prep_col] + time_df[cook_col],
        atol=tolerance,
        equal_nan=False,
    )
)

total_mismatch_when_all_present = all_three_present & ~total_equals_prep_plus_cook

# Cases where cook is missing but prep + total exist
# If TotalTime == PrepTime, missing CookTime likely means CookTime = 0
prep_equals_total_when_cook_missing = (
    prep_total_present_cook_missing
    & np.isclose(
        time_df[prep_col],
        time_df[total_col],
        atol=tolerance,
        equal_nan=False,
    )
)

prep_total_mismatch_when_cook_missing = (
    prep_total_present_cook_missing
    & ~prep_equals_total_when_cook_missing
)

# Cases where prep is missing but cook + total exist
# If TotalTime == CookTime, missing PrepTime likely means PrepTime = 0
cook_equals_total_when_prep_missing = (
    cook_total_present_prep_missing
    & np.isclose(
        time_df[cook_col],
        time_df[total_col],
        atol=tolerance,
        equal_nan=False,
    )
)

cook_total_mismatch_when_prep_missing = (
    cook_total_present_prep_missing
    & ~cook_equals_total_when_prep_missing
)

# Cases where total is missing but prep + cook exist
# These are recoverable by TotalTime = PrepTime + CookTime
recoverable_total_missing = prep_cook_present_total_missing

time_consistency_summary = pd.DataFrame(
    {
        "count": [
            all_three_present.sum(),
            total_equals_prep_plus_cook.sum(),
            total_mismatch_when_all_present.sum(),
            prep_total_present_cook_missing.sum(),
            prep_equals_total_when_cook_missing.sum(),
            prep_total_mismatch_when_cook_missing.sum(),
            cook_total_present_prep_missing.sum(),
            cook_equals_total_when_prep_missing.sum(),
            cook_total_mismatch_when_prep_missing.sum(),
            recoverable_total_missing.sum(),
        ],
        "percent_total_dataset": [
            all_three_present.mean() * 100,
            total_equals_prep_plus_cook.mean() * 100,
            total_mismatch_when_all_present.mean() * 100,
            prep_total_present_cook_missing.mean() * 100,
            prep_equals_total_when_cook_missing.mean() * 100,
            prep_total_mismatch_when_cook_missing.mean() * 100,
            cook_total_present_prep_missing.mean() * 100,
            cook_equals_total_when_prep_missing.mean() * 100,
            cook_total_mismatch_when_prep_missing.mean() * 100,
            recoverable_total_missing.mean() * 100,
        ],
        "percent_within_relevant_case": [
            100.0,
            total_equals_prep_plus_cook.sum() / all_three_present.sum() * 100
            if all_three_present.sum() > 0 else np.nan,
            total_mismatch_when_all_present.sum() / all_three_present.sum() * 100
            if all_three_present.sum() > 0 else np.nan,
            100.0,
            prep_equals_total_when_cook_missing.sum() / prep_total_present_cook_missing.sum() * 100
            if prep_total_present_cook_missing.sum() > 0 else np.nan,
            prep_total_mismatch_when_cook_missing.sum() / prep_total_present_cook_missing.sum() * 100
            if prep_total_present_cook_missing.sum() > 0 else np.nan,
            100.0,
            cook_equals_total_when_prep_missing.sum() / cook_total_present_prep_missing.sum() * 100
            if cook_total_present_prep_missing.sum() > 0 else np.nan,
            cook_total_mismatch_when_prep_missing.sum() / cook_total_present_prep_missing.sum() * 100
            if cook_total_present_prep_missing.sum() > 0 else np.nan,
            100.0,
        ],
        "interpretation": [
            "All three time fields exist",
            "TotalTime equals PrepTime + CookTime",
            "TotalTime does not equal PrepTime + CookTime",
            "CookTime missing, but PrepTime and TotalTime exist",
            "PrepTime equals TotalTime; missing CookTime can likely be resolved as 0",
            "PrepTime and TotalTime differ; missing CookTime can be derived as TotalTime - PrepTime if nonnegative",
            "PrepTime missing, but CookTime and TotalTime exist",
            "CookTime equals TotalTime; missing PrepTime can likely be resolved as 0",
            "CookTime and TotalTime differ; missing PrepTime can be derived as TotalTime - CookTime if nonnegative",
            "TotalTime missing, but PrepTime and CookTime exist; total can be derived",
        ],
    },
    index=[
        "all_three_present",
        "total_equals_prep_plus_cook",
        "total_mismatch_when_all_present",
        "cook_missing_prep_total_present",
        "prep_equals_total_when_cook_missing",
        "prep_total_mismatch_when_cook_missing",
        "prep_missing_cook_total_present",
        "cook_equals_total_when_prep_missing",
        "cook_total_mismatch_when_prep_missing",
        "recoverable_total_missing",
    ],
)

time_consistency_summary["count"] = time_consistency_summary["count"].astype(int)
time_consistency_summary["percent_total_dataset"] = time_consistency_summary[
    "percent_total_dataset"
].round(4)
time_consistency_summary["percent_within_relevant_case"] = time_consistency_summary[
    "percent_within_relevant_case"
].round(4)

display(time_consistency_summary)

# Optional: save output
# time_consistency_summary.to_csv("artifacts/week5/eda_summaries/time_consistency_summary.csv")


# ============================================================
# 4. Optional: create masks for later imputation decisions
# ============================================================

# CookTime can be derived if missing and TotalTime >= PrepTime
can_derive_cooktime = (
    time_df[cook_col].isna()
    & time_df[prep_col].notna()
    & time_df[total_col].notna()
    & (time_df[total_col] >= time_df[prep_col])
)

# PrepTime can be derived if missing and TotalTime >= CookTime
can_derive_preptime = (
    time_df[prep_col].isna()
    & time_df[cook_col].notna()
    & time_df[total_col].notna()
    & (time_df[total_col] >= time_df[cook_col])
)

# TotalTime can be derived if missing and prep/cook exist
can_derive_totaltime = (
    time_df[total_col].isna()
    & time_df[prep_col].notna()
    & time_df[cook_col].notna()
)

derivation_summary = pd.DataFrame(
    {
        "count": [
            can_derive_cooktime.sum(),
            can_derive_preptime.sum(),
            can_derive_totaltime.sum(),
        ],
        "percent_total_dataset": [
            can_derive_cooktime.mean() * 100,
            can_derive_preptime.mean() * 100,
            can_derive_totaltime.mean() * 100,
        ],
        "rule": [
            "CookTime = TotalTime - PrepTime",
            "PrepTime = TotalTime - CookTime",
            "TotalTime = PrepTime + CookTime",
        ],
    },
    index=[
        "can_derive_cooktime",
        "can_derive_preptime",
        "can_derive_totaltime",
    ],
)

derivation_summary["count"] = derivation_summary["count"].astype(int)
derivation_summary["percent_total_dataset"] = derivation_summary[
    "percent_total_dataset"
].round(4)

display(derivation_summary)

# Optional: save output
# derivation_summary.to_csv("artifacts/week5/eda_summaries/time_derivation_summary.csv")

,count,percent_total_dataset,interpretation
servings_only,260852,49.9222,"RecipeServings exists, RecipeYield missing"
yield_only,95692,18.3137,"RecipeYield exists, RecipeServings missing"
both_present,78754,15.0720,Both RecipeServings and RecipeYield exist
both_missing,87219,16.6921,Both RecipeServings and RecipeYield missing
total_recipes,522517,100.0000,Total number of recipes


,count,percent_total_dataset,interpretation
all_three_present,439971,84.2022,"Prep, cook, and total time are all present"
prep_cook_present_total_missing,0,0.0000,"Prep and cook exist, but total time is missing"
prep_total_present_cook_missing,82544,15.7974,"Prep and total exist, but cook time is missing"
cook_total_present_prep_missing,1,0.0002,"Cook and total exist, but prep time is missing"
only_prep_present,0,0.0000,Only prep time exists
only_cook_present,0,0.0000,Only cook time exists
only_total_present,0,0.0000,Only total time exists
all_three_missing,1,0.0002,All three time fields are missing
total_recipes,522517,100.0000,Total number of recipes


,count,percent_total_dataset,percent_within_relevant_case,interpretation
all_three_present,439971,84.2022,100.0000,All three time fields exist
total_equals_prep_plus_cook,439947,84.1976,99.9945,TotalTime equals PrepTime + CookTime
total_mismatch_when_all_present,24,0.0046,0.0055,TotalTime does not equal PrepTime + CookTime
cook_missing_prep_total_present,82544,15.7974,100.0000,"CookTime missing, but PrepTime and TotalTime e..."
prep_equals_total_when_cook_missing,81962,15.6860,99.2949,PrepTime equals TotalTime; missing CookTime ca...
prep_total_mismatch_when_cook_missing,582,0.1114,0.7051,PrepTime and TotalTime differ; missing CookTim...
prep_missing_cook_total_present,1,0.0002,100.0000,"PrepTime missing, but CookTime and TotalTime e..."
cook_equals_total_when_prep_missing,0,0.0000,0.0000,CookTime equals TotalTime; missing PrepTime ca...
cook_total_mismatch_when_prep_missing,1,0.0002,100.0000,CookTime and TotalTime differ; missing PrepTim...
recoverable_total_missing,0,0.0000,100.0000,"TotalTime missing, but PrepTime and CookTime e..."


,count,percent_total_dataset,rule
can_derive_cooktime,82543,15.7972,CookTime = TotalTime - PrepTime
can_derive_preptime,1,0.0002,PrepTime = TotalTime - CookTime
can_derive_totaltime,0,0.0000,TotalTime = PrepTime + CookTime


## Leakage audit table
We mark outcome and popularity fields to avoid leakage into content features.

In [10]:
leakage_rows = [
    {"column": "AggregatedRating", "layer": "outcome-popularity", "allowed_for_content_features": False, "reason": "Derived from user behavior; leakage risk"},
    {"column": "ReviewCount", "layer": "outcome-popularity", "allowed_for_content_features": False, "reason": "Popularity metric; leakage risk"},
    {"column": "Rating", "layer": "interaction", "allowed_for_content_features": False, "reason": "Outcome signal from reviews; reserve for evaluation"},
    {"column": "RecipeId", "layer": "identifier", "allowed_for_content_features": False, "reason": "Identifier only"},
    {"column": "ReviewId", "layer": "identifier", "allowed_for_content_features": False, "reason": "Identifier only"},
    {"column": "AuthorId", "layer": "identifier", "allowed_for_content_features": False, "reason": "Identifier only"},
    {"column": "RecipeIngredientParts", "layer": "text-list", "allowed_for_content_features": True, "reason": "Core content signal"},
    {"column": "RecipeKeywords", "layer": "text-list", "allowed_for_content_features": True, "reason": "Tag-like content signal"},
    {"column": "RecipeCategory", "layer": "categorical", "allowed_for_content_features": True, "reason": "Content category"},
    {"column": "DatePublished", "layer": "temporal", "allowed_for_content_features": True, "reason": "Temporal metadata; use carefully"}
]
leakage_audit = pd.DataFrame(leakage_rows)
save_df(leakage_audit, ARTIFACT_DIR / "leakage_audit.csv")
display(leakage_audit)

,column,layer,allowed_for_content_features,reason
0,AggregatedRating,outcome-popularity,False,Derived from user behavior; leakage risk
1,ReviewCount,outcome-popularity,False,Popularity metric; leakage risk
2,Rating,interaction,False,Outcome signal from reviews; reserve for evalu...
3,RecipeId,identifier,False,Identifier only
4,ReviewId,identifier,False,Identifier only
5,AuthorId,identifier,False,Identifier only
6,RecipeIngredientParts,text-list,True,Core content signal
7,RecipeKeywords,text-list,True,Tag-like content signal
8,RecipeCategory,categorical,True,Content category
9,DatePublished,temporal,True,Temporal metadata; use carefully


**Interpretation**
- AggregatedRating, ReviewCount, and review Rating are outcomes and must be excluded from intrinsic content features.
- These fields are appropriate for evaluation or popularity baselines only.

## Numeric feature EDA for PCA
We summarize candidate numeric features and evaluate missingness, skewness, and correlation.

In [11]:
numeric_candidates = [
    "Calories", "FatContent", "SaturatedFatContent", "CholesterolContent",
    "SodiumContent", "CarbohydrateContent", "FiberContent", "SugarContent",
    "ProteinContent", "RecipeServings", "NumIngredients", "NumQuantities",
    "CookTime_Minutes", "PrepTime_Minutes", "TotalTime_Minutes"
]
numeric_cols = [c for c in numeric_candidates if c in recipes.columns]

summary_rows = []
for col in numeric_cols:
    s = pd.to_numeric(recipes[col], errors="coerce")
    missing_pct = s.isna().mean() * 100
    s_clean = s.dropna()
    if s_clean.empty:
        continue
    percentiles = s_clean.quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).to_dict()
    summary_rows.append({
        "column": col,
        "count": int(s_clean.count()),
        "missing_pct": float(round(missing_pct, 2)),
        "mean": float(s_clean.mean()),
        "std": float(s_clean.std()),
        "min": float(s_clean.min()),
        "p1": float(percentiles.get(0.01, np.nan)),
        "p5": float(percentiles.get(0.05, np.nan)),
        "p25": float(percentiles.get(0.25, np.nan)),
        "p50": float(percentiles.get(0.50, np.nan)),
        "p75": float(percentiles.get(0.75, np.nan)),
        "p95": float(percentiles.get(0.95, np.nan)),
        "p99": float(percentiles.get(0.99, np.nan)),
        "max": float(s_clean.max()),
        "skewness": float(s_clean.skew())
    })

numeric_summary = pd.DataFrame(summary_rows)
save_df(numeric_summary, ARTIFACT_DIR / "numeric_feature_summary.csv")
display(numeric_summary.head())

if not numeric_summary.empty:
    plt.figure(figsize=(10, 5))
    plt.bar(numeric_summary["column"], numeric_summary["missing_pct"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Missing %")
    plt.title("Numeric Feature Missingness")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "numeric_missingness_bar.png")
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.bar(numeric_summary["column"], numeric_summary["skewness"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Skewness")
    plt.title("Numeric Feature Skewness")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "numeric_skewness_bar.png")
    plt.close()

if numeric_cols:
    numeric_df = recipes[numeric_cols].apply(pd.to_numeric, errors="coerce")
    numeric_df = numeric_df.fillna(numeric_df.median(numeric_only=True))
    corr = numeric_df.corr()
    corr_df = corr.reset_index().rename(columns={"index": "column"})
    save_df(corr_df, ARTIFACT_DIR / "numeric_correlation_matrix.csv")

    plt.figure(figsize=(8, 6))
    plt.imshow(corr, cmap="coolwarm", interpolation="nearest")
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Numeric Feature Correlation Heatmap")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "numeric_correlation_heatmap.png")
    plt.close()

time_cols = [c for c in ["TotalTime_Minutes", "PrepTime_Minutes", "CookTime_Minutes"] if c in recipes.columns]
time_consistency = []
if len(time_cols) == 3:
    total = pd.to_numeric(recipes["TotalTime_Minutes"], errors="coerce")
    prep = pd.to_numeric(recipes["PrepTime_Minutes"], errors="coerce")
    cook = pd.to_numeric(recipes["CookTime_Minutes"], errors="coerce")
    time_consistency = [
        {"check": "total_lt_prep", "count": int((total < prep).sum())},
        {"check": "total_lt_cook", "count": int((total < cook).sum())},
        {"check": "any_negative", "count": int(((total < 0) | (prep < 0) | (cook < 0)).sum())}
    ]
else:
    time_consistency = [{"check": "time_columns_missing", "count": np.nan}]

time_consistency_df = pd.DataFrame(time_consistency)
save_df(time_consistency_df, ARTIFACT_DIR / "time_consistency_summary.csv")
display(time_consistency_df)

,column,count,missing_pct,mean,std,min,p1,p5,p25,p50,p75,p95,p99,max,skewness
0,Calories,522517,0.0,484.438580,1397.116649,0.0,7.3,55.0,174.2,317.1,529.1,1305.8,3642.672,612854.6,252.214451
1,FatContent,522517,0.0,24.614922,111.485798,0.0,0.0,0.2,5.6,13.8,27.4,75.6,208.100,64368.1,410.572600
2,SaturatedFatContent,522517,0.0,9.559457,46.622621,0.0,0.0,0.0,1.5,4.7,10.8,30.2,84.900,26740.6,409.774335
3,CholesterolContent,522517,0.0,86.487003,301.987009,0.0,0.0,0.0,3.8,42.6,107.9,291.5,684.484,130456.4,250.140140
4,SodiumContent,522517,0.0,767.263878,4203.620531,0.0,0.5,7.5,123.3,353.3,792.2,2217.1,5569.084,1246921.1,102.243265


,check,count
0,total_lt_prep,5
1,total_lt_cook,8
2,any_negative,0


**Interpretation**
- Strong skewness suggests log1p transforms or clipping for outliers before scaling.
- High correlations support PCA to reduce redundant nutrition and time features.
- If time inconsistencies are large, we may drop or cap those values.

## Ingredient list-text EDA for TF-IDF + TruncatedSVD
We parse ingredient lists, normalize tokens, and evaluate TF-IDF thresholds.

In [12]:
from collections import Counter

def safe_parse_list(value):
    if isinstance(value, list):
        return value, True
    if pd.isna(value):
        return [], False
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed, True
        except (ValueError, SyntaxError):
            return [], False
    return [], False

def normalize_token(token):
    token = str(token).lower().strip()
    token = re.sub(r"[^a-z0-9 ]+", " ", token)
    token = re.sub(r" +", " ", token).strip()
    token = token.replace(" ", "_")
    return token

ingredient_col = find_column(recipes, ["RecipeIngredientParts", "Ingredients", "IngredientParts"])

if ingredient_col is None:
    print("Ingredient column not found.")
else:
    parsed = recipes[ingredient_col].apply(safe_parse_list)
    ingredient_lists = parsed.apply(lambda x: x[0])
    parse_success_rate = parsed.apply(lambda x: x[1]).mean() * 100

    recipe_tokens = []
    ingredient_counts = []
    counter = Counter()

    for lst in ingredient_lists:
        tokens = [normalize_token(t) for t in lst]
        tokens = [t for t in tokens if t]
        token_set = set(tokens)
        recipe_tokens.append(token_set)
        ingredient_counts.append(len(token_set))
        counter.update(token_set)

    zero_ingredient_recipes = int((np.array(ingredient_counts) == 0).sum())
    avg_ingredients = float(np.mean(ingredient_counts)) if ingredient_counts else 0.0
    median_ingredients = float(np.median(ingredient_counts)) if ingredient_counts else 0.0
    p95_ingredients = float(np.percentile(ingredient_counts, 95)) if ingredient_counts else 0.0
    unique_tokens = len(counter)

    ingredient_summary = pd.DataFrame([
        {"metric": "parse_success_rate_pct", "value": round(parse_success_rate, 2)},
        {"metric": "recipes_with_zero_ingredients", "value": zero_ingredient_recipes},
        {"metric": "avg_ingredients_per_recipe", "value": round(avg_ingredients, 2)},
        {"metric": "median_ingredients_per_recipe", "value": round(median_ingredients, 2)},
        {"metric": "p95_ingredients_per_recipe", "value": round(p95_ingredients, 2)},
        {"metric": "unique_ingredient_tokens", "value": unique_tokens}
    ])
    save_df(ingredient_summary, ARTIFACT_DIR / "ingredient_parse_summary.csv")
    display(ingredient_summary)

    top_ingredients = pd.DataFrame(counter.most_common(50), columns=["ingredient", "recipe_count"])
    save_df(top_ingredients, ARTIFACT_DIR / "top_ingredients.csv")
    display(top_ingredients.head(10))

    top30 = top_ingredients.head(30).iloc[::-1]
    plt.figure(figsize=(8, 8))
    plt.barh(top30["ingredient"], top30["recipe_count"])
    plt.title("Top 30 Ingredients by Recipe Frequency")
    plt.xlabel("Recipe count")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "top_30_ingredients.png")
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.hist(ingredient_counts, bins=40)
    plt.title("Ingredients per Recipe")
    plt.xlabel("Unique ingredients per recipe")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "ingredients_per_recipe_hist.png")
    plt.close()

    min_df_values = [2, 5, 10, 25, 50]
    tfidf_rows = []
    n_docs = len(recipe_tokens)
    for min_df in min_df_values:
        vocab = {t for t, df in counter.items() if df >= min_df}
        vocab_size = len(vocab)
        if n_docs == 0 or vocab_size == 0:
            coverage_count = 0
            coverage_pct = 0.0
            density = 0.0
        else:
            coverage_count = sum(1 for tokens in recipe_tokens if tokens & vocab)
            coverage_pct = (coverage_count / n_docs) * 100
            token_hits = sum(len(tokens & vocab) for tokens in recipe_tokens)
            density = token_hits / (n_docs * vocab_size)

        if vocab_size == 0:
            comment = "no vocabulary"
        elif coverage_pct < 80:
            comment = "coverage too low"
        elif vocab_size > 100000:
            comment = "vocab too large"
        else:
            comment = "reasonable tradeoff"

        tfidf_rows.append({
            "min_df": min_df,
            "vocab_size": vocab_size,
            "docs_with_tokens": coverage_count,
            "coverage_pct": round(coverage_pct, 2),
            "approx_density": round(density, 6),
            "comment": comment
        })

    tfidf_comparison = pd.DataFrame(tfidf_rows)
    save_df(tfidf_comparison, ARTIFACT_DIR / "tfidf_threshold_comparison.csv")
    display(tfidf_comparison)

,metric,value
0,parse_success_rate_pct,100.00
1,recipes_with_zero_ingredients,0.00
2,avg_ingredients_per_recipe,7.68
3,median_ingredients_per_recipe,7.00
4,p95_ingredients_per_recipe,14.00
5,unique_ingredient_tokens,7287.00


,ingredient,recipe_count
0,salt,190464
1,butter,123598
2,sugar,102808
3,onion,86321
4,eggs,80436
5,water,79884
6,olive_oil,72763
7,flour,59081
8,milk,58800
9,garlic_cloves,58612


,min_df,vocab_size,docs_with_tokens,coverage_pct,approx_density,comment
0,2,5681,522508,100.00,0.001352,reasonable tradeoff
1,5,4726,522489,99.99,0.001624,reasonable tradeoff
2,10,4052,522455,99.99,0.001892,reasonable tradeoff
3,25,3191,522341,99.97,0.002394,reasonable tradeoff
4,50,2529,522141,99.93,0.003003,reasonable tradeoff


**Interpretation**
- Ingredient lists are sparse, high-dimensional, and best represented with TF-IDF.
- TruncatedSVD is preferred over PCA for sparse matrices and scales better than dense PCA.
- The TF-IDF threshold comparison guides the min_df choice for vocabulary size vs coverage.

## Keyword list-text EDA
We repeat parsing and token summaries if keywords are available.

In [13]:
keyword_col = find_column(recipes, ["RecipeKeywords", "Keywords"])

if keyword_col is None:
    print("Keyword column not found.")
else:
    parsed_kw = recipes[keyword_col].apply(safe_parse_list)
    keyword_lists = parsed_kw.apply(lambda x: x[0])
    parse_success_rate_kw = parsed_kw.apply(lambda x: x[1]).mean() * 100

    kw_counter = Counter()
    kw_counts = []
    for lst in keyword_lists:
        tokens = [normalize_token(t) for t in lst]
        tokens = [t for t in tokens if t]
        token_set = set(tokens)
        kw_counts.append(len(token_set))
        kw_counter.update(token_set)

    keyword_summary = pd.DataFrame([
        {"metric": "parse_success_rate_pct", "value": round(parse_success_rate_kw, 2)},
        {"metric": "unique_keyword_tokens", "value": len(kw_counter)},
        {"metric": "avg_keywords_per_recipe", "value": round(float(np.mean(kw_counts)), 2)},
        {"metric": "median_keywords_per_recipe", "value": round(float(np.median(kw_counts)), 2)}
    ])
    save_df(keyword_summary, ARTIFACT_DIR / "keyword_parse_summary.csv")
    display(keyword_summary)

    top_keywords = pd.DataFrame(kw_counter.most_common(50), columns=["keyword", "recipe_count"])
    save_df(top_keywords, ARTIFACT_DIR / "top_keywords.csv")
    display(top_keywords.head(10))

,metric,value
0,parse_success_rate_pct,100.00
1,unique_keyword_tokens,314.00
2,avg_keywords_per_recipe,4.84
3,median_keywords_per_recipe,4.00


,keyword,recipe_count
0,easy,276838
1,60_mins,149589
2,30_mins,112229
3,4_hours,111071
4,meat,103191
5,15_mins,89340
6,healthy,83305
7,vegetable,81264
8,low_cholesterol,74121
9,beginner_cook,67092


**Interpretation**
- If keyword coverage and vocabulary are reasonable, keywords can be added as tokens to the TF-IDF content representation.
- If sparse or noisy, they may be excluded or treated separately.

## Categorical EDA
We summarize RecipeCategory to decide if it should be one-hot encoded or represented as a token.

In [14]:
category_col = find_column(recipes, ["RecipeCategory", "Category"])

if category_col is None:
    print("Category column not found.")
else:
    category_counts = recipes[category_col].fillna("Unknown").value_counts()
    total_recipes = len(recipes)
    top30 = category_counts.head(30)

    coverage_top10 = category_counts.head(10).sum() / total_recipes * 100
    coverage_top20 = category_counts.head(20).sum() / total_recipes * 100
    coverage_top30 = category_counts.head(30).sum() / total_recipes * 100
    rare_categories = int((category_counts < 50).sum())

    category_summary = pd.DataFrame([
        {"metric": "unique_categories", "value": int(category_counts.shape[0])},
        {"metric": "coverage_top10_pct", "value": round(coverage_top10, 2)},
        {"metric": "coverage_top20_pct", "value": round(coverage_top20, 2)},
        {"metric": "coverage_top30_pct", "value": round(coverage_top30, 2)},
        {"metric": "rare_categories_lt_50", "value": rare_categories}
    ])
    save_df(category_summary, ARTIFACT_DIR / "category_summary.csv")

    top_categories = top30.reset_index()
    top_categories.columns = ["category", "count"]
    save_df(top_categories, ARTIFACT_DIR / "top_categories.csv")
    display(category_summary)
    display(top_categories.head(10))

    plt.figure(figsize=(8, 6))
    plt.barh(top30.index[::-1], top30.values[::-1])
    plt.title("Top 30 Categories")
    plt.xlabel("Recipe count")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "top_30_categories.png")
    plt.close()

,metric,value
0,unique_categories,312.00
1,coverage_top10_pct,46.35
2,coverage_top20_pct,64.22
3,coverage_top30_pct,74.65
4,rare_categories_lt_50,104.00


,category,count
0,Dessert,62072
1,Lunch/Snacks,32586
2,One Dish Meal,31345
3,Vegetable,27231
4,Breakfast,21101
5,Beverages,16076
6,Chicken,13249
7,Meat,13131
8,Breads,12804
9,Pork,12603


**Interpretation**
- If many rare categories exist, grouping them as "Other" is appropriate for one-hot encoding.
- Alternatively, categories can be added as tokens to the content TF-IDF representation.

## Temporal EDA
We evaluate DatePublished for parsing success and basic distributions.

In [15]:
date_col = find_column(recipes, ["DatePublished", "Published", "Date"])

if date_col is None:
    print("DatePublished column not found.")
else:
    parsed_dates = pd.to_datetime(recipes[date_col], errors="coerce")
    parse_success_rate = parsed_dates.notna().mean() * 100
    min_date = parsed_dates.min()
    max_date = parsed_dates.max()
    missing_dates = int(parsed_dates.isna().sum())

    recipes_by_year = parsed_dates.dt.year.value_counts().sort_index()

    temporal_summary = pd.DataFrame([
        {"metric": "parse_success_rate_pct", "value": round(parse_success_rate, 2)},
        {"metric": "min_date", "value": str(min_date)},
        {"metric": "max_date", "value": str(max_date)},
        {"metric": "missing_or_invalid_dates", "value": missing_dates}
    ])
    save_df(temporal_summary, ARTIFACT_DIR / "temporal_summary.csv")
    recipes_by_year_df = recipes_by_year.reset_index()
    recipes_by_year_df.columns = ["year", "recipe_count"]
    save_df(recipes_by_year_df, ARTIFACT_DIR / "recipes_by_year.csv")
    display(temporal_summary)

    plt.figure(figsize=(8, 5))
    plt.plot(recipes_by_year.index, recipes_by_year.values, marker="o")
    plt.title("Recipes by Year")
    plt.xlabel("Year")
    plt.ylabel("Recipe count")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "recipes_by_year.png")
    plt.close()

,metric,value
0,parse_success_rate_pct,100.0
1,min_date,1999-08-06 00:40:00+00:00
2,max_date,2020-12-22 22:12:00+00:00
3,missing_or_invalid_dates,0


**Interpretation**
- Temporal features like published_year, published_month, published_dayofweek, or recipe_age_years are possible.
- Temporal features should be used carefully to avoid over-clustering by era rather than culinary similarity.

## Interaction EDA for future recommender
We analyze reviews as an interaction layer and avoid joining them to recipe-level PCA features.

In [ ]:
rating_col = find_column(reviews, ["Rating", "rating"])

if not (author_id_col and review_recipe_id_col):
    print("Required review columns not found.")
else:
    user_counts = reviews.groupby(author_id_col).size()
    recipe_counts = reviews.groupby(review_recipe_id_col).size()
    n_users = int(user_counts.shape[0])
    n_reviewed_recipes = int(recipe_counts.shape[0])
    n_recipes_catalog = int(recipes[recipe_id_col].nunique()) if recipe_id_col else n_reviewed_recipes
    observed_interactions = int(len(reviews))
    total_pairs = n_users * n_recipes_catalog
    density = (observed_interactions / total_pairs) if total_pairs else 0

    pct_users_one = (user_counts == 1).mean() * 100
    pct_users_ge5 = (user_counts >= 5).mean() * 100
    pct_recipes_ge5 = (recipe_counts >= 5).mean() * 100

    recipes_with_reviews = set(reviews[review_recipe_id_col].dropna().astype(str))
    recipes_catalog = set(recipes[recipe_id_col].dropna().astype(str)) if recipe_id_col else set()
    recipes_with_zero_reviews = len(recipes_catalog - recipes_with_reviews) if recipes_catalog else np.nan

    interaction_summary = pd.DataFrame([
        {"metric": "num_users", "value": n_users},
        {"metric": "num_reviewed_recipes", "value": n_reviewed_recipes},
        {"metric": "num_recipes_catalog", "value": n_recipes_catalog},
        {"metric": "observed_interactions", "value": observed_interactions},
        {"metric": "total_possible_pairs", "value": total_pairs},
        {"metric": "user_item_density", "value": density},
        {"metric": "pct_users_with_one_review", "value": round(pct_users_one, 2)},
        {"metric": "pct_users_with_ge5_reviews", "value": round(pct_users_ge5, 2)},
        {"metric": "pct_recipes_with_ge5_reviews", "value": round(pct_recipes_ge5, 2)},
        {"metric": "recipes_with_zero_reviews", "value": recipes_with_zero_reviews}
    ])
    save_df(interaction_summary, ARTIFACT_DIR / "interaction_summary.csv")
    display(interaction_summary)

    if rating_col:
        rating_dist = reviews[rating_col].value_counts().sort_index().reset_index()
        rating_dist.columns = ["rating", "count"]
    else:
        rating_dist = pd.DataFrame(columns=["rating", "count"])
    save_df(rating_dist, ARTIFACT_DIR / "rating_distribution.csv")

    if rating_col and not rating_dist.empty:
        plt.figure(figsize=(6, 4))
        plt.bar(rating_dist["rating"].astype(str), rating_dist["count"])
        plt.title("Rating Distribution")
        plt.xlabel("Rating")
        plt.ylabel("Count")
        plt.tight_layout()
        plt.savefig(FIGURE_DIR / "rating_distribution.png")
        plt.close()

    user_summary = pd.DataFrame([
        {"metric": "mean", "value": float(user_counts.mean())},
        {"metric": "median", "value": float(user_counts.median())},
        {"metric": "p95", "value": float(user_counts.quantile(0.95))}
    ])
    save_df(user_summary, ARTIFACT_DIR / "reviews_per_user_summary.csv")

    plt.figure(figsize=(6, 4))
    plt.hist(user_counts, bins=50)
    plt.xscale("log")
    plt.title("Reviews per User (log x-scale)")
    plt.xlabel("Reviews per user")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "reviews_per_user_hist_log.png")
    plt.close()

    recipe_summary = pd.DataFrame([
        {"metric": "mean", "value": float(recipe_counts.mean())},
        {"metric": "median", "value": float(recipe_counts.median())},
        {"metric": "p95", "value": float(recipe_counts.quantile(0.95))}
    ])
    save_df(recipe_summary, ARTIFACT_DIR / "reviews_per_recipe_summary.csv")

    plt.figure(figsize=(6, 4))
    plt.hist(recipe_counts, bins=50)
    plt.xscale("log")
    plt.title("Reviews per Recipe (log x-scale)")
    plt.xlabel("Reviews per recipe")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "reviews_per_recipe_hist_log.png")
    plt.close()

,metric,value
0,num_users,2.719070e+05
1,num_reviewed_recipes,2.716780e+05
2,num_recipes_catalog,5.225170e+05
3,observed_interactions,1.401982e+06
4,total_possible_pairs,1.420760e+11
5,user_item_density,9.867829e-06
6,pct_users_with_one_review,7.348000e+01
7,pct_users_with_ge5_reviews,1.018000e+01
8,pct_recipes_with_ge5_reviews,2.386000e+01
9,recipes_with_zero_reviews,2.508430e+05


**Interpretation**
- Reviews form a sparse user-item interaction matrix and are appropriate for collaborative filtering later.
- Reviews should not be joined into recipe-level PCA or clustering features.

## Final feature-design recommendation table
We summarize decisions for each feature group to guide the reproducible pipeline.

In [ ]:
# feature_recs = pd.DataFrame([
#     {
#         "feature_group": "numeric_nutrition_time_count",
#         "columns_or_source": "Calories, FatContent, ProteinContent, Time, NumIngredients, NumQuantities",
#         "observed_issue": "missingness and skewness",
#         "preprocessing_decision": "impute, optional log1p or clipping, scale",
#         "dimensionality_method": "PCA",
#         "downstream_use": "content similarity and clustering"
# ,
# ,
# ,
# ,
# ,
# ,
# ])

# save_df(feature_recs, ARTIFACT_DIR / "feature_design_recommendations.csv")
# display(feature_recs)

## Final technical conclusion
- Numeric recipe features should be imputed, optionally log1p-transformed or clipped for skew/outliers, scaled, and then reduced with PCA.
- Ingredient list-text features should be represented with TF-IDF and reduced with TruncatedSVD due to sparse high-dimensional structure.
- RecipeCategory can be one-hot encoded with rare categories grouped as Other, or added as tokens in the content representation.
- DatePublished supports temporal features but should not dominate culinary similarity.
- AggregatedRating, ReviewCount, and review Rating are outcomes and must not be used as intrinsic content features.
- Reviews remain an interaction layer for later recommendation evaluation and should not be joined into recipe-level PCA or clustering.